# Gemma 4 31B Arabic Agentic SFT — Cloud Stable

Fine-tunes Gemma 4 31B into an Arabic-speaking agentic model on **OneClickAMD.ai**.

**Capabilities:**
- Arabic conversation (MSA فصحى + Egyptian dialect العامية المصرية)
- Translation (AR ↔ EN)
- Information gathering and report generation in Arabic
- Tool calling / function calling with Arabic context

**Cloud stability features** (from proven Gemma 4 31B pipeline):
- Chunked training (600 steps/chunk)
- HF Hub checkpoint sync after each chunk
- Auto-resume from cloud checkpoints on VM restart
- LoRA rank 16 for stability
- Unsloth gradient checkpointing


### Installation


In [ ]:
# === Create a clean AMD/Unsloth kernel (doc-aligned path) ===
# This follows the official AMD + Unsloth guidance: isolated env, ROCm PyTorch, then Unsloth.
import os, re, shutil, subprocess, sys
from pathlib import Path
VENV = os.path.expanduser('~/unsloth_gemma4_env')
PY = os.path.join(VENV, 'bin', 'python')
PIP = [PY, '-m', 'pip']

def run(cmd):
    print('\n$ ' + ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip():
        out = r.stdout.strip().splitlines()
        for line in out[-8:]:
            print(line)
    if r.returncode != 0:
        err = r.stderr.strip() or '(no stderr)'
        raise RuntimeError(err)

def detect_rocm_tag():
    candidates = []
    try:
        r = subprocess.run(['amd-smi', 'version'], capture_output=True, text=True)
        candidates.append(r.stdout)
    except FileNotFoundError:
        pass
    try:
        candidates.append(Path('/opt/rocm/.info/version').read_text())
    except Exception:
        pass
    try:
        r = subprocess.run(['hipconfig', '--version'], capture_output=True, text=True)
        candidates.append(r.stdout)
    except FileNotFoundError:
        pass
    for text in candidates:
        m = re.search(r'(?:ROCm version: |HIP version: )([0-9]+)\.([0-9]+)', text)
        if not m:
            m = re.search(r'([0-9]+)\.([0-9]+)', text)
        if m:
            return f'rocm{m.group(1)}.{m.group(2)}'
    return 'rocm7.0'

rocm_tag = detect_rocm_tag()
print('ROCm wheel tag:', rocm_tag)

if os.path.isdir(VENV):
    shutil.rmtree(VENV)
run([sys.executable, '-m', 'venv', VENV])
run(PIP + ['install', '--upgrade', 'pip', 'setuptools', 'wheel', 'ipykernel'])
run(PIP + ['install', '--upgrade', '--force-reinstall', 'torch', 'torchvision', 'torchaudio', '--index-url', f'https://download.pytorch.org/whl/{rocm_tag}'])
run(PIP + ['install', '--no-deps', 'unsloth', 'unsloth-zoo'])
run(PIP + ['install', '--no-deps', 'git+https://github.com/unslothai/unsloth-zoo.git'])
run(PIP + ['install', 'unsloth[amd] @ git+https://github.com/unslothai/unsloth'])
run(PIP + ['install', '--upgrade', 'git+https://github.com/huggingface/transformers.git'])
run(PIP + ['install', '--upgrade', 'timm'])
run([PY, '-m', 'ipykernel', 'install', '--user', '--name', 'unsloth-gemma4-amd', '--display-name', 'Python (unsloth-gemma4-amd)'])

print('\nKernel created: Python (unsloth-gemma4-amd)')
print('Next: Kernel -> Change Kernel -> Python (unsloth-gemma4-amd)')
print('Then restart kernel and run Cell 3.')


In [ ]:
# === Verify you are on the clean Unsloth AMD kernel ===
import os, sys
from pathlib import Path
VENV = os.environ.get('UNSLOTH_VENV', str(Path.home() / 'unsloth_gemma4_env'))
print('sys.executable:', sys.executable)
print('sys.prefix:', sys.prefix)
print('VIRTUAL_ENV:', os.environ.get('VIRTUAL_ENV'))
on_expected_kernel = (
    sys.prefix == VENV
    or sys.executable.startswith(VENV + os.sep)
    or os.environ.get('VIRTUAL_ENV') == VENV
)
if not on_expected_kernel:
    raise RuntimeError(
        'Wrong kernel. Change kernel to Python (unsloth-gemma4-amd), then restart and re-run this cell.'
    )

import unsloth, unsloth_zoo, huggingface_hub, regex, transformers
from transformers import AutoConfig

print(f'unsloth: {getattr(unsloth, "__version__", "OK")} ({unsloth.__file__})')
print(f'unsloth_zoo: {getattr(unsloth_zoo, "__version__", "OK")} ({unsloth_zoo.__file__})')
print(f'huggingface_hub: {huggingface_hub.__version__} ({huggingface_hub.__file__})')
print(f'transformers: {transformers.__version__} ({transformers.__file__})')
c = AutoConfig.from_pretrained("unsloth/gemma-4-31B-it")
print(f'AutoConfig OK: model_type={c.model_type}')

import torch; torch._dynamo.config.recompile_limit = 64
print('Next: run the HF token cell, then the Load Model cell.')


In [ ]:
import os
from pathlib import Path
import huggingface_hub

# ==========================================================
# ⚠️ PASTE YOUR HUGGING FACE TOKEN BETWEEN THE QUOTES BELOW:
# ==========================================================
HF_TOKEN = ""

HF_REPO_ID = "mtita/gemma4-31b-arabic-agent-lora"

# Fallback to environment variable if the string above is empty
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN":
    print("⚠️  No HF token set. Hub pushing disabled, and downloading gated models may fail!")
    print("   Please paste your token in the HF_TOKEN variable above.")
    HF_TOKEN = ""
    PUSH_TO_HUB = False
else:
    print("✅ Hugging Face Token recognized!")
    huggingface_hub.login(token=HF_TOKEN)
    PUSH_TO_HUB = True

# === Training Output ===
TRAINING_OUTPUT_DIR = os.environ.get(
    "TRAINING_OUTPUT_DIR",
    str(Path.home() / "gemma4_runs" / "gemma4_arabic_agent_sft")
)
Path(TRAINING_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("HF repo:", HF_REPO_ID)
print("Push enabled:", PUSH_TO_HUB)
print("Training outputs:", TRAINING_OUTPUT_DIR)


### Load Model


In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-31B-it",
    dtype = torch.bfloat16,
    max_seq_length = 4096,
        packing = True,
    load_in_4bit = False,
    full_finetuning = False,
    token = HF_TOKEN or None,
)

### Add LoRA Adapters


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Text-only
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_gradient_checkpointing = "unsloth",
)


### Data Prep

Using the `gemma-4` non-thinking chat template. Gemma 4 renders conversations like:
```
<bos><|turn>user
مرحباً<turn|>
<|turn>model
أهلاً!<turn|>
```

**Gemma 4 data best practices (from Unsloth docs):**
1. Standard roles: system, user, assistant
2. No `tool` role — fold into user messages
3. `.removeprefix('<bos>')` since processor adds it
4. Loss of 13-15 is normal for E2B/E4B; 31B should be 1-3


In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)


In [ ]:
from datasets import load_dataset

# Load Arabic agentic dataset from HF Hub
DATASET_SPLIT = "train"
dataset = load_dataset(
    "mtita/gemma4-arabic-agent-training",
    data_files="arabic_agentic_dataset.jsonl",
    split=DATASET_SPLIT,
)
print(f"Loaded {len(dataset)} training examples")
print(dataset[0]["messages"][:2])


Map roles for Gemma 4 (system → user prefix, tool → user prefix) and apply chat template.


In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        formatted = []
        for msg in messages:
            role = msg["role"]
            content = msg.get("content", "") or ""
            if role == "system":
                sys_msg = {"role": "user", "content": f"[System Instructions]\n{content}"}
                ack_msg = {"role": "assistant", "content": "مفهوم."}
                if formatted and formatted[-1]["role"] == "user":
                    formatted[-1]["content"] += f"\n\n{sys_msg['content']}"
                    formatted.append(ack_msg)
                else:
                    formatted.append(sys_msg)
                    formatted.append(ack_msg)
            elif role == "user":
                if formatted and formatted[-1]["role"] == "user":
                    formatted[-1]["content"] += f"\n\n{content}"
                else:
                    formatted.append({"role": "user", "content": content})
            elif role == "assistant":
                if formatted and formatted[-1]["role"] == "assistant":
                    formatted[-1]["content"] += f"\n\n{content}"
                else:
                    formatted.append({"role": "assistant", "content": content})
            elif role == "tool":
                tool_name = msg.get("name", "tool")
                tool_text = f"[نتيجة الأداة: {tool_name}]\n{content}"
                if formatted and formatted[-1]["role"] == "user":
                    formatted[-1]["content"] += f"\n\n{tool_text}"
                else:
                    formatted.append({"role": "user", "content": tool_text})
        # Ensure starts with user
        if not formatted or formatted[0]["role"] != "user":
            texts.append("")
            continue
        if formatted[-1]["role"] != "assistant":
            texts.append("")
            continue
        try:
            text = tokenizer.apply_chat_template(
                formatted, tokenize=False, add_generation_prompt=False
            ).removeprefix('<bos>')
            texts.append(text)
        except Exception:
            texts.append("")
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
dataset = dataset.filter(lambda x: len(x["text"]) > 50)
print(f"Formatted: {len(dataset)} examples")
print(dataset[0]["text"][:500])

### Stability Strategy

Training uses **chunked execution** (600 steps per chunk, 30 chunks max) with:
- Checkpoint save after each chunk
- Synchronous upload to HF Hub after each chunk
- Auto-resume from cloud checkpoints on kernel restart

If the kernel dies: re-run setup cells → this cell auto-resumes from last saved checkpoint.


### Train the model


In [ ]:
from transformers.trainer_utils import get_last_checkpoint

LAST_CHECKPOINT = get_last_checkpoint(TRAINING_OUTPUT_DIR)
print("Latest checkpoint:", LAST_CHECKPOINT or "none")


Only train on assistant outputs, ignore user inputs:


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 4096,
        packing = True,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        num_train_epochs = 2,
        warmup_steps = 20,
        learning_rate = 1e-4,
        logging_steps = 1,
        optim = "adamw_torch",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        report_to = "none",
        output_dir = TRAINING_OUTPUT_DIR,
        save_strategy = "no",
        hub_model_id = HF_REPO_ID,
        push_to_hub = PUSH_TO_HUB,
        hub_strategy = "checkpoint",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)


### Verify masking before training

Run these to confirm only assistant responses are being trained on.


In [ ]:
# Verify masking - should see full conversation
tokenizer.decode(trainer.train_dataset[0]["input_ids"])


In [ ]:
# Verify masking - should see ONLY assistant responses (rest is spaces)
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")


### Start or resume training (chunked)

If the kernel dies or the notebook disconnects, re-open the notebook, run the setup cells again, then run the cell below. It will auto-resume from the last cloud checkpoint.


In [ ]:
import gc
import os
import json
import torch
from huggingface_hub import snapshot_download, HfApi
from transformers.trainer_utils import get_last_checkpoint

def get_step_from_checkpoint(ckpt_dir):
    if not ckpt_dir: return 0
    try:
        with open(os.path.join(ckpt_dir, "trainer_state.json")) as f:
            return json.load(f).get("global_step", 0)
    except:
        try: return int(ckpt_dir.split("-")[-1])
        except: return 0

# 0. Sync missing checkpoints from the Hub
if PUSH_TO_HUB:
    print(f"\n📥 Pulling rescued checkpoints from Hugging Face: {HF_REPO_ID}...")
    try:
        snapshot_download(
            repo_id=HF_REPO_ID,
            local_dir=TRAINING_OUTPUT_DIR,
            token=HF_TOKEN,
            allow_patterns=["*checkpoint*"]
        )
        print("✅ Cloud checkpoints restored to the fresh VM!")
    except Exception as e:
        print("No prior checkpoints found on Hub, starting fresh.")

    trainer.args.save_strategy = "steps"
    trainer.args.save_steps = 600
    trainer.args.save_total_limit = 2
    trainer.args.push_to_hub = False

CHUNK_SIZE = 600
TOTAL_CHUNKS = 30

for i in range(TOTAL_CHUNKS):
    LAST_CHECKPOINT = get_last_checkpoint(TRAINING_OUTPUT_DIR)
    
    last_hf_ckpt = os.path.join(TRAINING_OUTPUT_DIR, "last-checkpoint")
    if not LAST_CHECKPOINT and os.path.exists(last_hf_ckpt):
        LAST_CHECKPOINT = last_hf_ckpt

    current_max_steps = (i + 1) * CHUNK_SIZE
    trainer.args.max_steps = current_max_steps
    
    completed_steps = get_step_from_checkpoint(LAST_CHECKPOINT)
    if LAST_CHECKPOINT and completed_steps >= current_max_steps:
        print(f"⏭️ Skipping chunk {i+1} (Already completed in a previous run)")
        continue
        
    print(f"\n🚀 Running Chunk {i+1}: Training up to step {current_max_steps}...")
    
    try:
        trainer_stats = trainer.train(resume_from_checkpoint=LAST_CHECKPOINT)
    except Exception as e:
        print(f"Training interrupted or hit an error: {e}")
        break
        
    # Synchronous upload to Hub
    if PUSH_TO_HUB:
        print(f"📦 Chunk {i+1} finished! Force-uploading checkpoints to the Cloud...")
        api = HfApi(token=HF_TOKEN)
        api.upload_folder(
            folder_path=TRAINING_OUTPUT_DIR,
            repo_id=HF_REPO_ID,
            repo_type="model",
            allow_patterns=["checkpoint-*/*"],
            commit_message=f"Arabic agent SFT: synced at step {current_max_steps}"
        )
        print("☁️ Cloud upload complete!")
    
    gc.collect()
    torch.cuda.empty_cache()
    print(f"🧹 Chunk {i+1} memory flushed. Moving to next chunk.")


### Push final adapter to Hugging Face


In [ ]:
if PUSH_TO_HUB:
    trainer.push_to_hub()
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print("Pushed final training artifacts to", HF_REPO_ID)
else:
    print("Skipping push_to_hub")


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


### Arabic Inference Test


In [ ]:
# Arabic tool-calling test
messages = [{"role": "user",
"content": "لديك الأدوات التالية:\n"
"- search(query): البحث في الإنترنت\n"
"- translate(text, from_lang, to_lang): ترجمة النصوص\n"
"- summarize(text): تلخيص النصوص\n\n"
"ابحث عن آخر أخبار التقنية في العالم العربي وقدم لي تقريراً موجزاً."}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 512,
                    temperature = 0.7, top_p = 0.9)

In [ ]:
# Arabic translation test
messages_tr = [{"role": "user",
"content": "ترجم الجملة التالية إلى الإنجليزية:\n\n"
"الذكاء الاصطناعي يغير طريقة عملنا وحياتنا اليومية بشكل جذري."}]

inputs_tr = tokenizer.apply_chat_template(
    messages_tr, tokenize=True, add_generation_prompt=True,
    return_tensors="pt", return_dict=True,
).to("cuda")

_ = model.generate(**inputs_tr, streamer=text_streamer, max_new_tokens=256,
                    temperature=0.3, top_p=0.9)


### Save LoRA adapters


In [ ]:
model.save_pretrained("gemma4_31b_arabic_lora")
tokenizer.save_pretrained("gemma4_31b_arabic_lora")

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print("Pushed LoRA adapter to", HF_REPO_ID)
else:
    print("Saved locally. Set HF_TOKEN to push to Hub.")

### Export to GGUF

Supported: q4_k_m (recommended), q5_k_m, q8_0, f16, bf16 and more.


In [ ]:
model.save_pretrained_gguf(
    "gemma4_31b_arabic_gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)


In [ ]:
# Push GGUF to HuggingFace
if PUSH_TO_HUB:
    model.push_to_hub_gguf(
        "mtita/gemma4-31b-arabic-agent-q4_k_m-GGUF",
        tokenizer,
        quantization_method = "q4_k_m",
        token = HF_TOKEN,
    )
    print("Pushed GGUF to Hub")
else:
    print("Saved GGUF locally. Set HF_TOKEN to push.")